**AI-powered paraphrasing tool**

Develop an AI-powered paraphrasing tool as a Python application or module. The tool should:

    ● Take a block of input text.

    ● Use deep learning (preferably transformer models like T5, BERT, GPT, or similar via Hugging Face Transformers).

    ● Generate a paraphrased version preserving meaning, improving clarity, and ensuring originality.

    ● Include built-in checks for grammar, spelling, and fluency of output.

Recommended technologies:

    ● Python, with Hugging Face Transformers and relevant NLP packages (spaCy,NLTK, etc.).

    ● TensorFlow or PyTorch for any model fine-tuning.

    ● Use only console or script-based input/output (no GUI or web interface).

    ● Optional: Integrate evaluation metrics like BLEU, ROUGE, or semantic similarity scores.

Expected Output:

    ● Well-commented Python scripts/notebooks.

    ● Sample results demonstrating paraphrased outputs.

    ● Evaluation report showing tool accuracy and originality.

In [1]:
#Install required libraries.

!pip install transformers torch sentencepiece
!pip install language-tool-python nltk rouge-score
!pip install sentence-transformers scikit-learn

In [2]:
#Import all required libraries.

import warnings
warnings.filterwarnings("ignore")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import language_tool_python

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util

In [3]:
#Load pretrained T5 paraphrasing model.

model_name = "humarin/chatgpt_paraphraser_on_T5_base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cpu")
model = model.to(device)
model.eval()

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [4]:
#Initialize grammar correction tool.

tool = language_tool_python.LanguageTool('en-US')

In [5]:
#Text Cleaning Function.

def clean_output(text):
    """
    Remove unwanted tokens and extra spaces.
    """
    unwanted = ["paraphrase:", "rewrite:", "instruction:", "input:"]

    for w in unwanted:
        text = text.replace(w, "")

    return " ".join(text.split()).strip()

#Load similarity model.

similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

#Paraphrasing Function.

def paraphrase_text(text):
    """
    Generate multiple paraphrases and select the best one
    based on semantic similarity and originality.
    """

    input_text = "paraphrase: " + text + " </s>"

    encoding = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    #Generate multiple candidates.
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=128,
        num_beams=8,
        num_return_sequences=5,
        do_sample=False,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3
    )

    #Decode outputs.
    candidates = [
        clean_output(tokenizer.decode(o, skip_special_tokens=True))
        for o in outputs
    ]

    best_text = None
    best_score = -1

    for c in candidates:

        #Semantic similarity.
        embeddings = similarity_model.encode(
            [text, c],
            convert_to_tensor=True
        )
        similarity = util.cos_sim(embeddings[0], embeddings[1]).item()

        #Originality(new words introduced).
        originality = len(set(c.split()) - set(text.split()))

        #Normalize originality.
        originality_score = originality / len(text.split())

        #Penalize near-copy outputs.
        if similarity > 0.97:
            similarity -= 0.2

        #Final score.
        final_score = 0.7 * similarity + 0.3 * originality_score

        if final_score > best_score:
            best_score = final_score
            best_text = c

    return best_text

#Grammar Correction Function.

def correct_grammar(text):
    """
    Improve grammar and sentence structure.
    """
    matches = tool.check(text)
    return language_tool_python.utils.correct(text, matches)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
#Take input from user.

input_text = input("Enter text to paraphrase: ")

Enter text to paraphrase: Deep learning models have significantly enhanced the performance of image and speech recognition systems by leveraging large amounts of labeled data.


In [7]:
#Generate paraphrase and apply grammar correction.

paraphrased = paraphrase_text(input_text)
final_output = correct_grammar(paraphrased)

print("\nOriginal Text:\n", input_text)
print("\nParaphrased Output:\n", final_output)


Original Text:
 Deep learning models have significantly enhanced the performance of image and speech recognition systems by leveraging large amounts of labeled data.

Paraphrased Output:
 The utilization of massive quantities of labeled data by deep learning models has resulted in significant improvements in both image and speech recognition systems.


In [14]:
#BLEU Score.

reference = input_text.split()
candidate = final_output.split()

smooth = SmoothingFunction().method1

bleu_score = sentence_bleu(
    [reference],
    candidate,
    smoothing_function=smooth
)

print("\nBLEU Score:", round(bleu_score, 4))

#ROUGE Score.

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

rouge_scores = scorer.score(input_text, final_output)

print("\nROUGE Scores:")
print("ROUGE-1:", round(rouge_scores['rouge1'].fmeasure, 4))
print("ROUGE-2:", round(rouge_scores['rouge2'].fmeasure, 4))
print("ROUGE-L:", round(rouge_scores['rougeL'].fmeasure, 4))

#Semantic Similarity.

embeddings = similarity_model.encode([input_text, final_output])

embeddings = model_sim.encode([input_text, final_output])

similarity = cosine_similarity(
    [embeddings[0]],
    [embeddings[1]]
)

print("\nSemantic Similarity Score:", round(similarity[0][0], 4))


BLEU Score: 0.1407

ROUGE Scores:
ROUGE-1: 0.6222
ROUGE-2: 0.3721
ROUGE-L: 0.3556

Semantic Similarity Score: 0.9113


In [15]:
print("\n===== FINAL EVALUATION REPORT =====")

print("\n1. BLEU Score:")
print("Measures word overlap between original and paraphrased text.")
print("Lower to moderate score indicates better paraphrasing (less copying).")

print("\n2. ROUGE Scores:")
print("Measures how much important content is retained.")
print("Higher scores indicate better content preservation.")

print("\n3. Semantic Similarity:")
print("Measures meaning similarity using embeddings.")
print("High score confirms the paraphrase preserves the original meaning.")

print("\n4. Overall Conclusion:")
print("The model generates fluent, grammatically correct, and meaningful paraphrases.")
print("It effectively balances originality and semantic accuracy.")


===== FINAL EVALUATION REPORT =====

1. BLEU Score:
Measures word overlap between original and paraphrased text.
Lower to moderate score indicates better paraphrasing (less copying).

2. ROUGE Scores:
Measures how much important content is retained.
Higher scores indicate better content preservation.

3. Semantic Similarity:
Measures meaning similarity using embeddings.
High score confirms the paraphrase preserves the original meaning.

4. Overall Conclusion:
The model generates fluent, grammatically correct, and meaningful paraphrases.
It effectively balances originality and semantic accuracy.
